## Import 

In [59]:
import onnx
import lenet

print(f'onnx version="{onnx.__version__}" opset="{onnx.defs.onnx_opset_version()}"')

onnx version="1.23.2" opset="28"


In [60]:
onnx_model = onnx.load(lenet.onnx_filename)
onnx.checker.check_model(onnx_model)
print(f"onnx op_set version={onnx.defs.onnx_opset_version()}")
print(f"onnx_model.op_set_version={onnx_model.opset_import[0].version}")
print(f"onnx_model.ir_version={onnx_model.ir_version}")

model_input = onnx_model.graph.input[0]
print(f"onnx_model.input: name={model_input.name}, type={model_input.type.tensor_type.elem_type} shape={model_input.type.tensor_type.shape.dim}")

model_output = onnx_model.graph.output[0]
print(f"onnx_model.output: name={model_output.name}, type={model_output.type.tensor_type.elem_type} shape={model_output.type.tensor_type.shape.dim}")



onnx op_set version=28
onnx_model.op_set_version=20
onnx_model.ir_version=10
onnx_model.input: name=images, type=1 shape=[dim_value: 1
, dim_value: 1
, dim_value: 28
, dim_value: 28
]
onnx_model.output: name=logits, type=1 shape=[dim_value: 1
, dim_value: 10
]


# Initializer

In [61]:
for initializer in onnx_model.graph.initializer:
    dtype = TensorProto.DataType.Name(initializer.data_type)
    print(f"initializer: name={initializer.name}, type={dtype}, shape={initializer.dims}")

initializer: name=lenet_stack.0.weight, type=FLOAT, shape=[12, 1, 3, 3]
initializer: name=lenet_stack.0.bias, type=FLOAT, shape=[12]
initializer: name=lenet_stack.3.weight, type=FLOAT, shape=[32, 12, 3, 3]
initializer: name=lenet_stack.3.bias, type=FLOAT, shape=[32]
initializer: name=lenet_stack.7.weight, type=FLOAT, shape=[120, 1152]
initializer: name=lenet_stack.7.bias, type=FLOAT, shape=[120]
initializer: name=lenet_stack.9.weight, type=FLOAT, shape=[84, 120]
initializer: name=lenet_stack.9.bias, type=FLOAT, shape=[84]
initializer: name=lenet_stack.11.weight, type=FLOAT, shape=[10, 84]
initializer: name=lenet_stack.11.bias, type=FLOAT, shape=[10]
initializer: name=val_3, type=INT64, shape=[2]


# Inspecting graph

In [62]:
for node in onnx_model.graph.node:
    print(f"node: name={node.name}, op_type={node.op_type}, input={node.input}, output={node.output}")

node: name=node_conv2d, op_type=Conv, input=['images', 'lenet_stack.0.weight', 'lenet_stack.0.bias'], output=['conv2d']
node: name=node_relu, op_type=Relu, input=['conv2d'], output=['relu']
node: name=node_avg_pool2d, op_type=AveragePool, input=['relu'], output=['avg_pool2d']
node: name=node_conv2d_1, op_type=Conv, input=['avg_pool2d', 'lenet_stack.3.weight', 'lenet_stack.3.bias'], output=['conv2d_1']
node: name=node_relu_1, op_type=Relu, input=['conv2d_1'], output=['relu_1']
node: name=node_avg_pool2d_1, op_type=AveragePool, input=['relu_1'], output=['avg_pool2d_1']
node: name=node_view, op_type=Reshape, input=['avg_pool2d_1', 'val_3'], output=['view']
node: name=node_linear, op_type=Gemm, input=['view', 'lenet_stack.7.weight', 'lenet_stack.7.bias'], output=['linear']
node: name=node_relu_2, op_type=Relu, input=['linear'], output=['relu_2']
node: name=node_linear_1, op_type=Gemm, input=['relu_2', 'lenet_stack.9.weight', 'lenet_stack.9.bias'], output=['linear_1']
node: name=node_sigmoi